# AfriQBench Notebook 06 — IBM Hardware Benchmark

This notebook takes the fixed four-qubit transverse-field Ising exam from Notebooks 01 and 02 and scores it on IBM hardware. The local check runs first. An IBM job is sent only when `RUN_HARDWARE` is set to `True`.

The benchmark Hamiltonian is

$$H=-J\sum_{i=1}^{N-1} Z_i Z_{i+1}-h\sum_{i=1}^{N} X_i.$$

For this exam, $N=4$, $J=1$, $h=1$, and the chain is open, so the ends are not joined:

$$H=-(Z_0Z_1+Z_1Z_2+Z_2Z_3)-(X_0+X_1+X_2+X_3).$$

The exact ground-state energy and the ideal-circuit energy stay separate answer keys. The hardware score is how far the chip's energy lands from those keys, together with the shot count, the compiled circuit depth, and the two-qubit gate count.

## 1. Environment

From the repository root:

```bash
python -m pip install -e ".[quantum,notebook]"
```

The IBM settings belong in `.env` at the repository root. The notebook loads that file and does not print the key.

```bash
export QISKIT_IBM_TOKEN="your-api-key"
export QISKIT_IBM_INSTANCE="your-instance-crn"
export QISKIT_IBM_CHANNEL="ibm_quantum_platform"
```

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from IPython.display import Image, display

from afriqbench.quantum.hardware import (
    ANSATZ_ENERGY,
    CANONICAL_SHOTS,
    EXACT_ENERGY,
    fetch_ibm_job,
    format_report,
    load_project_env,
    measurement_specs,
    run_ibm_hardware,
    run_local_preflight,
    save_scorecard,
)


## 2. Formulas used in the score

Each Pauli term is measured separately. A $ZZ$ term is read in the computational basis. An $X$ term is read after a Hadamard on that qubit. With $S$ shots, the expectation of one term is

$$\langle P_k\rangle=\frac{1}{S}\sum_{s=1}^{S}\lambda_s,\qquad \lambda_s=\pm 1.$$

The coefficients are $c_k=-J$ for each neighbor pair and $c_k=-h$ for each transverse field. The estimated energy and its shot uncertainty are

$$E=\sum_k c_k\langle P_k\rangle,\qquad
\sigma_E=\sqrt{\sum_k c_k^2\,\sigma_k^2},\qquad
\sigma_k=\sqrt{\frac{1-\langle P_k\rangle^2}{S}}.$$

Two gaps are reported. The energy error compares the run with the exact ground state. The execution error compares the run with the ideal fixed circuit, whose energy is already $0.0012226231$ above the exact ground state.

$$\Delta_{\mathrm{exact}}=\lvert E_{\mathrm{run}}-E_{\mathrm{exact}}\rvert,\qquad
\Delta_{\mathrm{exec}}=\lvert E_{\mathrm{run}}-E_{\mathrm{circuit}}\rvert,\qquad
\delta=\frac{\Delta_{\mathrm{exact}}}{\lvert E_{\mathrm{exact}}\rvert}.$$

The stored answer keys for $N=4$, $J=1$, $h=1$ are

$$E_{\mathrm{exact}}=-4.7587704831,\qquad E_{\mathrm{circuit}}=-4.7575478600.$$

Circuit depth and the two-qubit gate count are taken after IBM compiles the circuit. Those compiled numbers are the hardware cost. The raw circuit, before compilation, has 6 two-qubit gates.

## 3. The seven measurement terms

Four sites and an open boundary give three neighbor pairs and four transverse-field terms. Every coefficient is $-1$ because $J=h=1$.

In [ ]:
specs = measurement_specs()
for spec in specs:
    print(
        f"{spec['term']:<6}  basis={spec['basis']}  "
        f"qubits={spec['qubits']}  coefficient={spec['coefficient']}"
    )
print("Exact energy:  ", f"{EXACT_ENERGY:.10f}")
print("Circuit energy:", f"{ANSATZ_ENERGY:.10f}")
print("Canonical shots per term:", CANONICAL_SHOTS)


## 4. Local preflight

This check rebuilds the fixed circuit and compares it with both answer keys. It uses a statevector, so there are no shots and no call to IBM. The circuit energy must stay at $-4.7575478600$, and the fidelity to the exact ground state must stay above $0.999$.

In [ ]:
preflight = run_local_preflight()
print(format_report(preflight))


## 5. Scorecard figure

The figure has four panels:

1. energy of the exact reference, the ideal circuit, and this run;
2. $\Delta_{\mathrm{exact}}$ and $\Delta_{\mathrm{exec}}$;
3. circuit depth of each measured term;
4. two-qubit gate count of each measured term.

Shots, uncertainty, and the relative error $\delta$ are printed along the bottom. On the local preflight the shot count is blank, because a statevector is not a measurement.

In [ ]:
json_path, figure_path = save_scorecard(
    preflight,
    repo_root / "results" / "hardware",
)
print("Scorecard:", json_path)
print("Figure:   ", figure_path)
display(Image(filename=str(figure_path)))


## 6. IBM account

Load `.env` before a hardware submission. Missing names are reported. The token itself is not printed.

In [ ]:
import os

load_project_env(repo_root)
required = ("QISKIT_IBM_TOKEN", "QISKIT_IBM_INSTANCE", "QISKIT_IBM_CHANNEL")
for name in required:
    print(f"{name}: {'set' if os.environ.get(name) else 'missing'}")


## 7. Submit the hardware job

Leave `RUN_HARDWARE` as `False` until the local preflight above has passed. Setting it to `True` sends seven circuits, one per Hamiltonian term, through the Qiskit Runtime Sampler.

`SHOTS = 1024` is a pilot budget. The stored simulator scorecards use $S=20000$ shots per term, so the total shot count at that budget is

$$S_{\mathrm{total}}=7\times 20000=140000.$$

The returned scorecard uses the same formulas as Section 2. Depth and two-qubit gate counts are the compiled values for the selected chip. Set `BACKEND` to a device name, or leave it as `None` to use the least busy chip with at least four qubits.

In [ ]:
RUN_HARDWARE = False
SHOTS = 1024
BACKEND = None

if RUN_HARDWARE:
    hardware = run_ibm_hardware(
        shots=SHOTS,
        backend_name=BACKEND,
        output_dir=repo_root / "results" / "hardware",
    )
    print(format_report(hardware))
    hardware_json, hardware_figure = save_scorecard(
        hardware,
        repo_root / "results" / "hardware",
    )
    print("Scorecard:", hardware_json)
    display(Image(filename=str(hardware_figure)))
else:
    print(
        "Hardware submission is off. Set RUN_HARDWARE = True "
        "after the local preflight passes."
    )


## 8. Resume a queued job

If the session closes while IBM still holds the job, paste the printed job id into `FETCH_JOB_ID` and run this cell. The pending file saved at submission supplies the compiled depth and gate counts. The counts that come back are converted with the energy formula in Section 2.

In [ ]:
FETCH_JOB_ID = ""

if FETCH_JOB_ID:
    resumed = fetch_ibm_job(
        FETCH_JOB_ID,
        output_dir=repo_root / "results" / "hardware",
    )
    print(format_report(resumed))
    resumed_json, resumed_figure = save_scorecard(
        resumed,
        repo_root / "results" / "hardware",
    )
    print("Scorecard:", resumed_json)
    display(Image(filename=str(resumed_figure)))
else:
    print("No job id set. Paste one into FETCH_JOB_ID to resume a submitted run.")


## 9. How to read the result

A completed hardware scorecard is graded only against the same molecule: $N=4$, $J=1$, $h=1$, open chain.

| Recorded quantity | Formula or source |
|---|---|
| Energy | $E=\sum_k c_k\langle P_k\rangle$ |
| Uncertainty | $\sigma_E$ from Section 2 |
| Energy error | $\Delta_{\mathrm{exact}}=\lvert E-E_{\mathrm{exact}}\rvert$ |
| Relative error | $\delta=\Delta_{\mathrm{exact}}/\lvert E_{\mathrm{exact}}\rvert$ |
| Execution error | $\Delta_{\mathrm{exec}}=\lvert E-E_{\mathrm{circuit}}\rvert$ |
| Shots | $S$ per term, and $7S$ in total |
| Circuit depth | compiled circuit for that chip |
| Two-qubit gates | compiled circuit for that chip |

The local preflight should show $\Delta_{\mathrm{exact}}=0.0012226231$ and $\Delta_{\mathrm{exec}}=0$. A hardware energy is then placed beside $-4.7587704831$. A smaller energy error is a closer run. Depth, two-qubit gates, and shots say what that closeness cost.